# Explorations for corpus building

If you want to select texts for your own corpus within the Project Gutenberg, this notebook can help you explore the subjects and titles that you find there in order to build your corpus.

In [ ]:
# @title Grant GoogleColab access to your GoogleDrive and import questions for this notebook
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add your module folder to Python path
import sys
module_path = f"/content/drive/My Drive/IDH/Notebooks"
sys.path.append(module_path)
print("GoogleColab can now access your GoogleDrive.")

In [ ]:
import os
import requests
import time
import pandas as pd
from collections import Counter
from pathlib import Path
from typing import List, Union

In [ ]:
# Get the PG metadata into a Pandas DataFrame
metapg = pd.read_csv(os.path.join(module_path, "PG/pg_catalog.csv"), index_col=0)

In [ ]:
# Flatten all subjects into a list
all_subjects = [
    subj.strip()
    for subjects in metapg['Subjects'].dropna()
    for subj in subjects.split(';')
]

# Count occurrences of each subject
subject_counter = Counter(all_subjects)

# Turn into a DataFrame
subjects_df = (
    pd.DataFrame(subject_counter.items(), columns=['Subjects', 'Count'])
    .sort_values(by="Count", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
up = 10
low = 9
subset_subjects = subjects_df[
    (subjects_df['Count'] >= low) & (subjects_df['Count'] <= up)
]

In [ ]:
import matplotlib.pyplot as plt

# Plot the top 20 subjects by count
plt.figure(figsize=(10,6))
plt.barh(subjects_df['Subjects'].head(20)[::-1], subjects_df['Count'].head(20)[::-1])
plt.xlabel("Number of Titles")
plt.ylabel("Subjects")
plt.title("Top 20 Subjects in Project Gutenberg Metadata")
plt.tight_layout()
plt.show()


In [ ]:
def filter_by_subject(df, col, topic):
    """
    Returns a dataframe with all books matching the given subject.
    """
    return df[df[col].str.contains(rf"\b{topic}\b", na=False)][['Title', 'Authors', 'Language']]

In [ ]:
# Suppose you want to get all titles under the subject "Biography"
bio = filter_by_subject(metapg, "Subjects", "Biography")

# Or you want titles that contain the word "Spain"
spain = filter_by_subject(metapg, "Title", "Spain")

In [ ]:
# You can use this cell to build your corpus
# Adapt the following code to match your selection
MY_CORPUS_NAME = filter_by_subject(metapg,
                                   col="SELECTED_COLUMN_NAME",
                                   topic="MY_TOPIC")

In [ ]:
def fetch_text_from_id(pgid: int, timeout=20) -> str:
    """
    Returns the text matching the input id.
    """
    url = f"https://www.gutenberg.org/cache/epub/{pgid}/pg{pgid}.txt"
    resp = requests.get(url, headers=headers, timeout=timeout)
    return resp.text

In [ ]:
# Use this cell to get the text ids
my_corpus = MY_CORPUS_NAME
ids = pd.Index(my_corpus.index).astype(int)

In [ ]:
# Define a directory where you will store the texts
# Write the right path
OUT_DIR = Path("WRITE_YOUR_PATH_HERE")

In [ ]:
# Make sure the output directory exists
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Loop over the IDs and scrape the corresponging texts
# The texts will be sotred in OUT_DIR that you defined above
for pgid in ids:
    text = fetch_text_from_id(int(pgid))
    out_path = OUT_DIR / f"pg{pgid}.txt"
    out_path.write_text(text, encoding="utf-8")
    print(f"pg{pgid}.txt has been downloaded")
    time.sleep(1)  # be polite to PG

In [ ]:
# Define headers
DEFAULT_HEADER = {
        "User-Agent": "Introduction to Digital Humanities 2025 (contact: mbednarkiewicz@faculty.ie.edu)",
        "Accept-Language": "en",
    }


def fetch_text_from_id(pgid: int, timeout: int = 20) -> str:
    """
    Returns the text matching the input Gutenberg ID.

    Raises:
        requests.exceptions.RequestException: If the request fails or returns a non-200 status.
    """
    url = f"https://www.gutenberg.org/cache/epub/{pgid}/pg{pgid}.txt"
    # Use DEFAULT_HEADER; if you defined your own 'header', pass it here.
    resp = requests.get(url, headers=DEFAULT_HEADER, timeout=timeout)
    resp.raise_for_status()
    return resp.text

In [ ]:
for pgid in ids:  # 'ids' must be defined
    try:
        text = fetch_text_from_id(int(pgid))
        out_path = OUT_DIR / f"pg{pgid}.txt"
        out_path.write_text(text, encoding="utf-8")
        print(f"pg{pgid}.txt downloaded successfully ({len(text)} chars)")
    except requests.exceptions.RequestException as e:
        print(f"Failed to download pg{pgid}.txt: {e}")
    except Exception as e:
        print(f"Unexpected error for pg{pgid}.txt: {e}")

    time.sleep(1)  # be polite to PG — keep this!